# FinSight Portfolio Optimization Engine

## Portfolio Optimization using Modern Portfolio Theory

This notebook documents the complete FinSight Wealth Platform portfolio optimization workflow.

### Objectives

- Load and inspect FinSight portfolio data
- Analyze historical stock prices
- Calculate asset returns and volatility
- Build a covariance matrix
- Analyze investor risk profiles
- Optimize portfolio allocation
- Calculate expected return
- Calculate portfolio volatility
- Calculate Sharpe ratio
- Calculate maximum drawdown
- Compare portfolios across risk profiles
- Visualize expected return versus risk

## 1. Project Overview

FinSight Wealth Platform is a FinTech portfolio optimization system designed for wealth management and robo-advisory use cases.

The system accepts an investor's risk profile and available capital and generates an optimized allocation across the available assets.

The project uses historical market data supplied by the client and applies quantitative portfolio optimization techniques.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Add project root to Python path
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

In [ ]:
from src.data_loader import (
    load_portfolio_data,
    load_risk_profiles
)

portfolio_df = load_portfolio_data()
risk_profiles = load_risk_profiles()

In [ ]:
print("Portfolio Dataset Shape:", portfolio_df.shape)

print("\nPortfolio Columns:")
print(portfolio_df.columns.tolist())

print("\nRisk Profile Dataset Shape:", risk_profiles.shape)

print("\nRisk Profile Columns:")
print(risk_profiles.columns.tolist())

## 2. Portfolio Dataset

The portfolio dataset contains historical prices for eight individual stocks and the NIFTY50 market index.

The individual assets used for optimization are:

- RELIANCE
- TCS
- HDFCBANK
- INFY
- ICICIBANK
- WIPRO
- BAJFINANCE
- HCLTECH

In [ ]:
assets = [
    "RELIANCE",
    "TCS",
    "HDFCBANK",
    "INFY",
    "ICICIBANK",
    "WIPRO",
    "BAJFINANCE",
    "HCLTECH"
]

portfolio_df["date"] = pd.to_datetime(
    portfolio_df["date"]
)

portfolio_df = portfolio_df.sort_values(
    "date"
)

prices = portfolio_df[assets]

print("Number of assets:", len(assets))
print("Number of observations:", len(prices))

prices.head()

## 3. Data Quality Check

Before performing financial calculations, the dataset is checked for:

- Missing values
- Duplicate rows
- Date ordering
- Basic descriptive statistics

In [ ]:
print("Missing Values:")
print(portfolio_df.isnull().sum())

print("\nDuplicate Rows:")
print(portfolio_df.duplicated().sum())

print("\nDate Range:")
print(
    portfolio_df["date"].min(),
    "to",
    portfolio_df["date"].max()
)

In [ ]:
prices.describe()

## 4. Historical Price Visualization

In [ ]:
plt.figure(figsize=(14, 7))

for asset in assets:
    plt.plot(
        portfolio_df["date"],
        portfolio_df[asset],
        label=asset
    )

plt.title("Historical Asset Prices")
plt.xlabel("Date")
plt.ylabel("Price")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

## 5. Daily Returns

Daily percentage returns are calculated using:

Daily Return = (Current Price / Previous Price) - 1

These returns are used for expected return, volatility, covariance and portfolio risk calculations.

In [ ]:
annual_expected_returns = returns.mean() * 252

annual_volatility = returns.std() * np.sqrt(252)

asset_performance = pd.DataFrame({
    "Expected Annual Return": annual_expected_returns,
    "Annual Volatility": annual_volatility
})

asset_performance["Expected Annual Return (%)"] = (
    asset_performance["Expected Annual Return"] * 100
)

asset_performance["Annual Volatility (%)"] = (
    asset_performance["Annual Volatility"] * 100
)

asset_performance

## 6. Asset Risk and Return

Expected annual return is estimated from the average daily return multiplied by 252 trading days.

Annual volatility is estimated from daily return standard deviation multiplied by the square root of 252.

In [ ]:
plt.figure(figsize=(10, 6))

plt.scatter(
    asset_performance["Annual Volatility (%)"],
    asset_performance["Expected Annual Return (%)"],
    s=100
)

for asset in assets:

    x = asset_performance.loc[
        asset,
        "Annual Volatility (%)"
    ]

    y = asset_performance.loc[
        asset,
        "Expected Annual Return (%)"
    ]

    plt.annotate(
        asset,
        (x, y),
        xytext=(5, 5),
        textcoords="offset points"
    )

plt.title("Asset Expected Return vs Risk")
plt.xlabel("Annual Volatility (%)")
plt.ylabel("Expected Annual Return (%)")
plt.grid(True)

plt.show()

## 7. Covariance Matrix

The covariance matrix measures how asset returns move relative to one another.

It is required for calculating portfolio-level volatility.

In [ ]:
covariance_matrix = returns.cov() * 252

covariance_matrix

In [ ]:
plt.figure(figsize=(10, 7))

sns.heatmap(
    returns.corr(),
    annot=True,
    fmt=".2f"
)

plt.title("Asset Correlation Matrix")
plt.tight_layout()

plt.show()

## 8. Investor Risk Profiles

The supplied FinSight risk-profile dataset contains four profiles:

- Conservative
- Moderate
- Aggressive
- Very Aggressive

Each profile contains an investment horizon, equity limits, expected return target and maximum drawdown target.

In [ ]:
risk_profiles

## 9. Portfolio Optimization

The optimization engine uses the available equity assets and attempts to maximize the Sharpe ratio while considering the target return and risk constraints defined by the investor profile.

Portfolio weights must sum to 100%.

Each individual stock has a maximum allocation of 40%.

In [ ]:
from src.optimizer import (
    optimize_portfolio,
    calculate_metrics,
    calculate_max_drawdown,
    get_profile
)

In [ ]:
profiles = [
    "conservative",
    "moderate",
    "aggressive",
    "very_aggressive"
]

optimization_results = []

for profile_name in profiles:

    weights = optimize_portfolio(
        profile_name
    )

    metrics = calculate_metrics(
        weights
    )

    max_drawdown = calculate_max_drawdown(
        weights
    )

    optimization_results.append({
        "Risk Profile": profile_name.replace(
            "_", " "
        ).title(),

        "Expected Return (%)":
            metrics["expected_return"] * 100,

        "Volatility (%)":
            metrics["volatility"] * 100,

        "Sharpe Ratio":
            metrics["sharpe_ratio"],

        "Maximum Drawdown (%)":
            max_drawdown * 100
    })

comparison_df = pd.DataFrame(
    optimization_results
)

comparison_df

## 10. Portfolio Allocations

In [ ]:
allocation_results = {}

for profile_name in profiles:

    weights = optimize_portfolio(
        profile_name
    )

    allocation_results[
        profile_name.replace(
            "_", " "
        ).title()
    ] = weights * 100

allocation_df = pd.DataFrame(
    allocation_results,
    index=assets
)

allocation_df

In [ ]:
allocation_df.plot(
    kind="bar",
    figsize=(14, 7)
)

plt.title("Portfolio Allocation by Risk Profile")
plt.xlabel("Asset")
plt.ylabel("Allocation (%)")
plt.xticks(rotation=45)
plt.legend(title="Risk Profile")
plt.tight_layout()

plt.show()

## 11. Expected Return vs Portfolio Risk

The following visualization compares the optimized portfolios using expected annual return and portfolio volatility.

In [ ]:
fig = px.scatter(
    comparison_df,
    x="Volatility (%)",
    y="Expected Return (%)",
    size="Sharpe Ratio",
    text="Risk Profile",
    title="Expected Return vs Portfolio Risk"
)

fig.update_traces(
    textposition="top center"
)

fig.show()

## 12. Sharpe Ratio

The Sharpe ratio measures risk-adjusted portfolio return.

The implementation uses a 6% risk-free rate:

Sharpe Ratio = (Portfolio Return - Risk Free Rate) / Portfolio Volatility

A higher Sharpe ratio indicates more return relative to the measured portfolio volatility within this model.

In [ ]:
comparison_df.sort_values(
    "Sharpe Ratio",
    ascending=False
)

## 13. Maximum Drawdown

Maximum drawdown measures the largest decline from a historical portfolio peak to a subsequent trough.

It provides an additional view of historical downside risk.

In [ ]:
for profile_name in profiles:

    weights = optimize_portfolio(
        profile_name
    )

    drawdown = calculate_max_drawdown(
        weights
    )

    print(
        f"{profile_name.replace('_', ' ').title():20}"
        f": {drawdown * 100:.2f}%"
    )

## 14. Conclusion

The FinSight portfolio optimization engine combines historical market analysis with quantitative portfolio optimization.

The completed workflow:

1. Loads client-provided financial datasets.
2. Validates and prepares historical price data.
3. Calculates daily and annualized asset returns.
4. Calculates asset volatility.
5. Builds covariance and correlation matrices.
6. Loads investor risk profiles.
7. Optimizes portfolio weights.
8. Calculates expected return, volatility and Sharpe ratio.
9. Calculates maximum drawdown.
10. Compares optimized portfolios across risk profiles.
11. Visualizes the relationship between portfolio return and risk.

The resulting notebook provides a documented quantitative workflow supporting the FinSight Wealth Platform dashboard and FastAPI service.